In [1]:
import sys
import pathlib as pl
import warnings

import pandas as pd
import numpy as np
import gtfs_kit as gk
import folium as fl

sys.path.append('../')

import make_gtfs as mg


DATA_DIR = pl.Path('../data')

%load_ext autoreload
%autoreload 2

warnings.filterwarnings(action='ignore')

/Users/danielbustillos/miniconda3/envs/analisis-general/lib/python3.10/site-packages/pandera/_pandas_deprecated.py:146: FutureWarning: Importing pandas-specific classes and functions from the
top-level pandera module will be **removed in a future version of pandera**.
If you're using pandera to validate pandas objects, we highly recommend updating
your import:

```
# old import
import pandera as pa

# new import
import pandera.pandas as pa
```

If you're using pandera to validate objects from other compatible libraries
like pyspark or polars, see the supported libraries section of the documentation
for more information on how to import pandera:

https://pandera.readthedocs.io/en/stable/supported_libraries.html

To disable this warning, set the environment variable:

```
export DISABLE_PANDERA_IMPORT_WARNING=True
```

  warnings.warn(_future_warning, FutureWarning)


In [2]:
path = "../../2-generacion-archivos-make_gtfs/data/proc/guadalajara/"
pfeed = mg.read_protofeed(path)
pfeed

ProtoFeed(meta=          agency_name                     agency_url   agency_timezone  \
0  guadalajara agency  https://guadalajaraagency.com  Pacific/Auckland   

  start_date  end_date  speed_route_type_3  speed_zone_type_2  
0   20250101  20250102                  30                 30  , service_windows=  service_window_id start_time  end_time  monday  tuesday  wednesday  \
0           weekday   06:00:00  09:00:00       1        1          1   

   thursday  friday  saturday  sunday  
0         1       1         1       1  , shapes=                             data.route.shortName  \
0               Puerto Vallarta_2_Puerto Vallarta   
1              Puerto Vallarta_11_Puerto Vallarta   
2                      Ruta Modelo_13_Ruta Modelo   
3              Puerto Vallarta_13_Puerto Vallarta   
4                                            _15_   
..                                            ...   
253  Línea 3 (Ruta 400)_SiTren_Línea 3 (Ruta 400)   
254                        Línea 1

In [3]:
sz = pfeed.speed_zones
display(sz)

m = fl.Map(tiles="CartoDB Positron")
fl.GeoJson(
    sz[lambda x: x.route_type == 3],
    tooltip=fl.GeoJsonTooltip(["speed_zone_id", "speed"])
).add_to(m)

bounds = sz.total_bounds
bounds = [(bounds[1], bounds[0]), (bounds[3], bounds[2])]  # rearrange for Folium
m.fit_bounds(bounds)
m

,route_type,speed_zone_id,speed,geometry
0,2,default-2,inf,"POLYGON ((-105.52108 20.43435, -105.52206 20.4..."
1,2,shape_basic_1,15.0,"POLYGON ((-105.52108 20.92041, -103.17045 20.9..."


AssertionError: The field speed_zone_id is not available in the data. Choose from: ().

In [9]:
feed = mg.build_feed(pfeed, stop_offset=100000, )

GEOSException: TopologyException: assigned depths do not match at 665321.46236410656 2293187.9993524496

In [ ]:
f = (
    feed.routes[['route_id', 'route_short_name']]
    .merge(pfeed.frequencies)
    .merge(pfeed.service_windows)
    .sort_values('route_id')
)
f

In [ ]:
# Map some trips
tids = [feed.trips.trip_id.iat[0], feed.trips.trip_id.iat[-1]]
feed.map_trips(tids, show_direction=True, show_stops=True)
    